In [1]:
import math
import os
import yaml
import torch
import json
import numpy as np
import pickle
from sklearn.metrics import f1_score

from daart.data import DataGenerator, compute_sequence_pad
from daart.transforms import ZScore
from daart_utils.data import DataHandler
import xgboost as xgb
from daart_utils.testtube import get_data_by_dtype

In [2]:
datas = {
    'fly': {
        'vids': [
            '2019_06_26_fly2',
            '2019_08_14_fly1',
            '2019_08_20_fly3',
            '2019_10_14_fly2',
            '2019_10_21_fly1',
        ],
        'parts': ['avg', 'still', 'walk', 'front_groom', 'back_groom', 'abdomen-move'],
        'sizes': [2,3,4,5],
        'ds_name': 'fly-5'
    },
    'oft': {
        'vids': [
            'OFT_39',
            'OFT_41',
            'OFT_43',
            'OFT_44',
            'OFT_49',
            'OFT_50',
            'OFT_51',
            'OFT_52',
            'OFT_54',
            'OFT_58',
        ],
        'parts': ['avg', 'supported', 'unsupported', 'grooming'],
        'sizes': [4,6,8,10],
        'ds_name': 'mouse-oft-aligned'
    } ,
    'ibl': {
        'vids': [
            'churchlandlab_CSHL045_2020-02-27-001',
            'cortexlab_KS020_2020-02-06-001',
            'hoferlab_SWC_043_2020-09-15-001',
            'mrsicflogellab_SWC_052_2020-10-22-001',
            'wittenlab_ibl_witten_27_2021-01-21-001',
        ],
        'parts': ['avg', 'still', 'move', 'wheel_turn', 'groom'],
        'sizes': [2,3,4,5],
        'ds_name': 'ibl'
    }, 
    'huga': {
        'vids': [
            'sess_06',
            'sess_08',
            'sess_11',
            'sess_13',
            'sess_17',
        ],
        'parts': ['avg', 'walking', 'running', 'going_up', 'going_down', 'sitting',
                     'sitting_down', 'standing_up', 'standing', 'down_elevator', 'up_elevator'],
        'sizes': [100,250,500,1000],
        'ds_name': 'huga'
    } 
}

input_dict = {
    'markers': 'm',
    'features-posvel': 'fp',
    'features-sturman': 'fs',
    'features-sturman-posvel': 'fspv'
}

data_path = '/home/bsb2144/daart_utils/data/'

In [6]:
ds = 'fly'
#ds = 'oft'
input_type = 'markers'
#input_type = 'features-sturman'
#input_type = 'features-posvel'

model_names = [
    'xgb',
    'rf'
]

save_names = [
    'xgb_{}'.format(input_dict[input_type]),
    'rf_{}'.format(input_dict[input_type]),
]


In [8]:
# note: for largest size v0 save states and latents
# loop over models
for mod_name, save_name in zip(model_names, save_names):
    # loop over data sizes
    sizes = datas[ds]['sizes']
    all_metrics = {}
    for size in sizes:
        # loop over versions
        size_metrics = []
        for v in range(5):
            # init model
            model_base = "/home/bsb2144/daart/results_daart/{}/multi-0/dtcn/".format(datas[ds]['ds_name'])
            if size == sizes[-1] and ds!='huga':
                model_dir = model_base + "{}-{}-good_sample-0_{}/version_{}".format(mod_name, size, input_type, v)
            else:
                model_dir = model_base + "{}-{}-good_sample-{}_{}/version_0".format(mod_name, size, v, input_type)
                
            model_file = os.path.join(model_dir, 'best_val_model.pt')
            arch_file = os.path.join(model_dir, 'hparams.yaml')
            with open(arch_file, 'rb') as f:
                hparams_new = yaml.safe_load(f)
            if hparams_new['model_class'] == 'xgboost':
                model_0 = xgb.Booster()
                model_0.load_model(model_file)
            elif hparams_new['model_class'] == 'random-forest':
                with open(model_file, 'rb') as f:
                    model_0 = pickle.load(f)
            else:
                raise NotImplementedError('"%s" is an invalid model typr' % hparams_new['model_class'])

            # loop over vids
            v_metrics = {
                'gt': [],
                'preds': []
            }
            for expt_id in datas[ds]['vids']:
                print(expt_id)
                # initialize data handler; point to correct base path
                handler = DataHandler(expt_id, base_path=os.path.join(data_path, datas[ds]['ds_name']))
                if input_type == 'markers':
                    markers_file = handler.get_marker_filepath()
                else:
                    markers_file = handler.get_feature_filepath(dirname=input_type)

                hand_labels_file = os.path.join(
                            "/home/bsb2144/daart/data/", datas[ds]['ds_name'], 'labels-hand', expt_id + '_labels.csv')

                # define data generator signals
                signals = ['markers', 'labels_strong']
                transforms = [ZScore(), None]
                paths = [markers_file, hand_labels_file]

                # build data generator
                data_gen_test = DataGenerator(
                    [expt_id], [signals], [transforms], [paths], device='cuda',#hparams['device'], 
                    batch_size=hparams_new['batch_size'], trial_splits='1;1;0;0', 
                    sequence_pad=hparams_new['sequence_pad'], sequence_length=hparams_new['sequence_length'],
                    input_type=hparams_new['input_type'])

                # load hand labels
                handler.load_hand_labels()
                states = np.argmax(handler.hand_labels.vals, axis=1)

                # compute predictions
                print('computing predictions for model 0...', end='')
                n_lags = 4
                X_test = data_gen_test.datasets[0].load_full_data(input_type=input_type)
                X_test = np.hstack([np.roll(X_test, i, axis=0) for i in range(-n_lags, n_lags + 1)])
                X_test = X_test[states>0]
                states = states[states>0]
                print('xt', X_test.shape)
                print('states', states.shape)

                if 'xgb' in mod_name:
                    dtest = xgb.DMatrix(X_test)
                    labels_pred = model_0.predict(dtest)
                    labels_model = np.argmax(labels_pred, axis=1) + 1
                else:
                    labels_model = model_0.predict(X_test)
                
                states = states[:len(labels_model)]
                print('states')
                unique, counts = np.unique(states, return_counts=True)
                print(np.asarray((unique, counts)).T)
                    
                print('labels_model', labels_model[:5], labels_model.shape)
                unique, counts = np.unique(labels_model, return_counts=True)
                print(np.asarray((unique, counts)).T)

                v_metrics['gt'] += list(states)
                v_metrics['preds'] += list(labels_model)
                
            # agg results over all vids (one version)
            all_gt = np.array(v_metrics['gt'])
            all_preds = np.array(v_metrics['preds'])
            f1_by_class = f1_score(all_preds[all_gt>0], all_gt[all_gt>0], average=None)
            print('f1_by_class', f1_by_class)
            f1_avg = np.mean(f1_by_class)
            temp_res = {'avg': f1_avg}
            for part, score in zip(datas[ds]['parts'][1:], f1_by_class):
                temp_res[part] = score
            size_metrics.append(temp_res)
            print('temp_res',temp_res)
            
        # agg results over all versions (one size)
        size_temp = {}
        for part in datas[ds]['parts']:
            size_temp[part] = {}
            mean_temp = np.mean([arr[part] for arr in size_metrics])
            sd_temp = np.std([arr[part] for arr in size_metrics])/len(sizes)
            size_temp[part]['mean'] = mean_temp
            size_temp[part]['sd'] = sd_temp
            
        # save size metrics to res dict
        all_metrics[size] = size_temp
            
    # save results
    print(all_metrics)
    save_path = '/home/bsb2144/daart/metrics/{}/{}.json'.format(ds, save_name)
    with open(save_path, 'w') as f:
        json.dump(all_metrics, f)
        print('saved file to {}'.format(f))
    
    
    
    

2019_06_26_fly2
NZ:  0
computing predictions for model 0...xt (1956, 144)
states (1956,)
states
[[  1 300]
 [  2 300]
 [  3 350]
 [  4 300]
 [  5 706]]
labels_model [5 5 5 5 5] (1956,)
[[  1 317]
 [  2 340]
 [  3 285]
 [  4 174]
 [  5 840]]
2019_08_14_fly1
NZ:  0
computing predictions for model 0...xt (1893, 144)
states (1893,)
states
[[  1 300]
 [  2 300]
 [  3 300]
 [  4 300]
 [  5 693]]
labels_model [5 5 5 1 5] (1893,)
[[  1  83]
 [  2 285]
 [  3 453]
 [  4 171]
 [  5 901]]
2019_08_20_fly3
NZ:  0
computing predictions for model 0...xt (1605, 144)
states (1605,)
states
[[  1 300]
 [  2 300]
 [  3 300]
 [  4 300]
 [  5 405]]
labels_model [5 5 5 5 5] (1605,)
[[  1 115]
 [  2 438]
 [  3 196]
 [  4 240]
 [  5 616]]
2019_10_14_fly2
NZ:  0
computing predictions for model 0...xt (1301, 144)
states (1301,)
states
[[  1 300]
 [  2 300]
 [  3 300]
 [  4 300]
 [  5 101]]
labels_model [3 3 3 3 3] (1301,)
[[  1 168]
 [  2 736]
 [  3 102]
 [  4 118]
 [  5 177]]
2019_10_21_fly1
NZ:  0
computing pre

NZ:  0
computing predictions for model 0...xt (1300, 144)
states (1300,)
states
[[  1 300]
 [  2 300]
 [  3 400]
 [  4 300]]
labels_model [2 2 2 2 2] (1300,)
[[  1 128]
 [  2 500]
 [  3 306]
 [  4 127]
 [  5 239]]
f1_by_class [0.37211493 0.59056231 0.74873737 0.68942548 0.76752768]
temp_res {'avg': 0.6336735530436655, 'still': 0.37211493170042387, 'walk': 0.5905623057360837, 'front_groom': 0.7487373737373737, 'back_groom': 0.6894254787676937, 'abdomen-move': 0.7675276752767527}
2019_06_26_fly2
NZ:  0
computing predictions for model 0...xt (1956, 144)
states (1956,)
states
[[  1 300]
 [  2 300]
 [  3 350]
 [  4 300]
 [  5 706]]
labels_model [4 5 5 5 5] (1956,)
[[  1 354]
 [  2 289]
 [  3 333]
 [  4 183]
 [  5 797]]
2019_08_14_fly1
NZ:  0
computing predictions for model 0...xt (1893, 144)
states (1893,)
states
[[  1 300]
 [  2 300]
 [  3 300]
 [  4 300]
 [  5 693]]
labels_model [4 4 2 1 3] (1893,)
[[  1 127]
 [  2 179]
 [  3 638]
 [  4 289]
 [  5 660]]
2019_08_20_fly3
NZ:  0
computing pr

NZ:  0
computing predictions for model 0...xt (1605, 144)
states (1605,)
states
[[  1 300]
 [  2 300]
 [  3 300]
 [  4 300]
 [  5 405]]
labels_model [5 5 5 5 5] (1605,)
[[  1 188]
 [  2 375]
 [  3 244]
 [  4 241]
 [  5 557]]
2019_10_14_fly2
NZ:  0
computing predictions for model 0...xt (1301, 144)
states (1301,)
states
[[  1 300]
 [  2 300]
 [  3 300]
 [  4 300]
 [  5 101]]
labels_model [1 1 1 1 1] (1301,)
[[  1 125]
 [  2 516]
 [  3 230]
 [  4 243]
 [  5 187]]
2019_10_21_fly1
NZ:  0
computing predictions for model 0...xt (1300, 144)
states (1300,)
states
[[  1 300]
 [  2 300]
 [  3 400]
 [  4 300]]
labels_model [2 2 2 2 2] (1300,)
[[  1 140]
 [  2 499]
 [  3 311]
 [  4 236]
 [  5 114]]
f1_by_class [0.61493289 0.68472181 0.83651399 0.82370481 0.82807641]
temp_res {'avg': 0.7575899825433268, 'still': 0.6149328859060403, 'walk': 0.6847218133647336, 'front_groom': 0.8365139949109415, 'back_groom': 0.8237048080506896, 'abdomen-move': 0.8280764104842292}
2019_06_26_fly2
NZ:  0
computing pre

FileNotFoundError: [Errno 2] No such file or directory: '/home/bsb2144/daart/results_daart/fly-5/multi-0/dtcn/xgb-5-good_sample-0_markers/version_0/hparams.yaml'

In [ ]:
print('done')